# Notebook de pruebas - ia-challenge

Este notebook sirve para probar paso a paso el pipeline del proyecto final.

Orden sugerido:

1. Validar rutas y dependencias.
2. Validar configuracion de base de datos sin mostrar contrasena.
3. Crear una SparkSession local.
4. Probar ingesta de una tabla.
5. Ejecutar ingesta completa.
6. Inspeccionar archivos Parquet generados.


In [1]:
import sys
print(sys.executable)
print(sys.version)

c:\Users\chris\Downloads\ia-challenge\.venv\Scripts\python.exe
3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))
print(PROJECT_ROOT)


c:\Users\chris\Downloads\ia-challenge


## 1. Validar configuracion de base de datos

Esta celda valida que `config/database.yml` exista y que `.env` tenga credenciales. No imprime la contrasena.


In [2]:
from src.utils.config_loader import get_jdbc_url, load_db_config

db_config = load_db_config(require_credentials=True)
print('Host:', db_config['host'])
print('Port:', db_config['port'])
print('Database:', db_config['database'])
print('Driver:', db_config['driver'])
print('User configured:', bool(db_config['user']))
print('Password configured:', bool(db_config['password']))
print('JDBC URL:', get_jdbc_url(db_config))


Host: www.bigdataybi.com
Port: 3306
Database: fake
Driver: com.mysql.cj.jdbc.Driver
User configured: True
Password configured: True
JDBC URL: jdbc:mysql://www.bigdataybi.com:3306/fake?useSSL=false&allowPublicKeyRetrieval=true


## 2. Crear SparkSession

Esta celda prueba que PySpark pueda iniciar con la configuracion del proyecto.


In [3]:
from src.utils.spark_session import create_spark_session

spark = create_spark_session('notebook_ia_challenge')
print('Spark version:', spark.version)


Spark version: 3.5.1


## 3. Probar lectura JDBC de una tabla

Primero se recomienda probar una tabla pequena como `shops` o `products`.


In [4]:
from src.ingest.extract_tables import read_source_table

sample_df = read_source_table(spark, 'shops')
print('Rows:', sample_df.count())
sample_df.printSchema()
sample_df.show(truncate=False)


2026-06-22 08:53:03,920 [INFO] src.ingest.extract_tables - Reading source table shops from www.bigdataybi.com:3306/fake


Rows: 3
root
 |-- id_sucursal: integer (nullable = true)
 |-- nombre_sucursal: string (nullable = true)
 |-- ciudad: string (nullable = true)
 |-- latitud: double (nullable = true)
 |-- longitud: double (nullable = true)
 |-- _loadtime: timestamp (nullable = true)

+-----------+------------------------+---------+-------+--------+-------------------+
|id_sucursal|nombre_sucursal         |ciudad   |latitud|longitud|_loadtime          |
+-----------+------------------------+---------+-------+--------+-------------------+
|3          |Sucursal Cuenca Centro  |Cuenca   |-2.8996|-79.0045|2025-06-26 19:34:50|
|2          |Sucursal Guayaquil Norte|Guayaquil|-2.1406|-79.8891|2025-06-26 19:34:50|
|1          |Sucursal Quito Sur      |Quito    |-0.251 |-78.5243|2025-06-26 19:34:50|
+-----------+------------------------+---------+-------+--------+-------------------+



## 4. Ingerir una tabla a Parquet raw

Esta celda escribe una tabla en `data/raw/<tabla>/` e incluye columnas de trazabilidad.


In [5]:
from src.ingest.extract_tables import ingest_table

rows = ingest_table('shops', spark=spark)
print('Rows ingested:', rows)


2026-06-22 08:53:18,388 [INFO] src.ingest.extract_tables - Reading source table shops from www.bigdataybi.com:3306/fake
2026-06-22 08:53:19,371 [INFO] src.ingest.extract_tables - Writing raw table shops to C:\Users\chris\Downloads\ia-challenge\data\raw\shops
2026-06-22 08:53:20,321 [INFO] src.ingest.extract_tables - Ingested 3 rows from shops


Rows ingested: 3


## 5. Ingerir todas las tablas fuente

Ejecuta la ingesta completa de `customers`, `products`, `sales` y `shops`.


In [6]:
from src.ingest.extract_tables import ingest_tables

counts = ingest_tables(spark=spark)
counts


2026-06-22 08:53:24,005 [INFO] src.ingest.extract_tables - Reading source table customers from www.bigdataybi.com:3306/fake
2026-06-22 08:53:24,978 [INFO] src.ingest.extract_tables - Writing raw table customers to C:\Users\chris\Downloads\ia-challenge\data\raw\customers
2026-06-22 08:53:25,561 [INFO] src.ingest.extract_tables - Ingested 10 rows from customers
2026-06-22 08:53:25,561 [INFO] src.ingest.extract_tables - Reading source table products from www.bigdataybi.com:3306/fake
2026-06-22 08:53:26,520 [INFO] src.ingest.extract_tables - Writing raw table products to C:\Users\chris\Downloads\ia-challenge\data\raw\products
2026-06-22 08:53:27,041 [INFO] src.ingest.extract_tables - Ingested 5 rows from products
2026-06-22 08:53:27,041 [INFO] src.ingest.extract_tables - Reading source table sales from www.bigdataybi.com:3306/fake
2026-06-22 08:53:27,965 [INFO] src.ingest.extract_tables - Writing raw table sales to C:\Users\chris\Downloads\ia-challenge\data\raw\sales
2026-06-22 08:53:28,50

{'customers': 10, 'products': 5, 'sales': 100, 'shops': 3}

## 6. Inspeccionar Parquet generado

Revisa las tablas crudas y las columnas de trazabilidad.


In [7]:
for table in ('customers', 'products', 'sales', 'shops'):
    path = PROJECT_ROOT / 'data' / 'raw' / table
    if path.exists():
        df = spark.read.parquet(str(path))
        print(f'{table}: {df.count()} rows')
        df.select('_source_system', '_source_database', '_source_table', '_extracted_at').show(3, truncate=False)
    else:
        print(f'{table}: pendiente, no existe {path}')


customers: 10 rows
+--------------+----------------+-------------+--------------------------+
|_source_system|_source_database|_source_table|_extracted_at             |
+--------------+----------------+-------------+--------------------------+
|mysql         |fake            |customers    |2026-06-22 08:53:24.986872|
|mysql         |fake            |customers    |2026-06-22 08:53:24.986872|
|mysql         |fake            |customers    |2026-06-22 08:53:24.986872|
+--------------+----------------+-------------+--------------------------+
only showing top 3 rows

products: 5 rows
+--------------+----------------+-------------+--------------------------+
|_source_system|_source_database|_source_table|_extracted_at             |
+--------------+----------------+-------------+--------------------------+
|mysql         |fake            |products     |2026-06-22 08:53:26.536013|
|mysql         |fake            |products     |2026-06-22 08:53:26.536013|
|mysql         |fake            |produc

## 7. Cerrar Spark

Ejecuta esta celda al final para liberar recursos.


In [8]:
spark.stop()
print('Spark stopped')


Spark stopped
